In [4]:
pip install pandas requests beautifulsoup4 lxml tqdm fake-useragent reportlab

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\ADMIN\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


*Import Libraries*

In [5]:
import requests
import pandas as pd
import time
import hashlib

from bs4 import BeautifulSoup
from tqdm import tqdm
from fake_useragent import UserAgent
from urllib.parse import urljoin

In [6]:
SEED_URL = [
    "https://www.education.go.ke/january-2025-school-re-opening",
    "https://kicd.ac.ke/features/curriculum-research/",
    "https://www.knec.ac.ke/",
    "https://www.helb.co.ke/",
    "https://www.universitiesfund.go.ke/",
    "https://www.kemi.ac.ke/",
    "https://www.unicef.org/kenya/",
    "https://redcross.or.ke/",
    "https://www.unesco.org/en/education",
    "https://accounts.ecitizen.go.ke/en/ministries/ministry-of-education?department=state-department-for-higher-education-and-research",
    "https://www.unesco.org/en/education",
    "https://pu.ac.ke/"
]

*Create Empty Cell List*

In [7]:
titles = []
dates = []
texts = []
urls = []
websites = []
hashes = []

*Create fake Browser Header*

In [8]:
from fake_useragent import UserAgent

ua = UserAgent()

headers = {
    "User-Agent": ua.random
}

*Web Request Function*

In [9]:
import urllib3
import requests
import time

from bs4 import BeautifulSoup
from fake_useragent import UserAgent
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Suppress SSL certificate warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Generate a random browser user-agent
ua = UserAgent()

EDUCATION_HEADERS = {
    "User-Agent": ua.random
}


def build_session():
    session = requests.Session()
    retry_strategy = Retry(
        total=3,
        connect=3,
        read=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update(EDUCATION_HEADERS)
    return session


EDUCATION_SESSION = build_session()


def fetch_education_page(url, timeout=10):
    """
    Downloads and parses an education webpage.
    Returns a BeautifulSoup object if successful,
    otherwise returns None.
    """

    try:
        response = EDUCATION_SESSION.get(
            url,
            timeout=(5, timeout),
            verify=False,
            allow_redirects=True
        )

        response.raise_for_status()

        content_type = response.headers.get("content-type", "").lower()
        if "text/html" not in content_type and "application/xhtml+xml" not in content_type:
            return None

        time.sleep(0.75)
        return BeautifulSoup(response.text, "html.parser")

    except requests.exceptions.Timeout:
        print(f"\n⏱️ Timed out while fetching: {url}")
        return None

    except requests.exceptions.SSLError as e:
        print(f"\n🔐 SSL error while fetching: {url}")
        print(f"Reason: {e}")
        return None

    except requests.exceptions.RequestException as e:
        print(f"\n❌ Failed to fetch: {url}")
        print(f"Reason: {e}")
        return None

*Discorver links*

In [10]:
from urllib.parse import urljoin

education_links = []

for website in tqdm(SEED_URL):

    soup = fetch_education_page(website)

    if soup is None:
        continue

    for link in soup.find_all("a", href=True):

        href = link["href"]

        # Convert relative URLs into absolute URLs
        full_url = urljoin(website, href)

        education_links.append(full_url)

print(f"Total education links discovered: {len(education_links)}")

100%|██████████| 12/12 [02:58<00:00, 14.86s/it]

Total education links discovered: 1996


*Education link*

In [11]:

ARTICLE_KEYWORDS = [

    "education",
    "news",
    "announcement",
    "announcements",
    "notice",
    "notices",
    "press",
    "press-release",
    "updates",
    "events",
    "schools",
    "school",
    "curriculum",
    "cbc",
    "university",
    "universities",
    "college",
    "tvet",
    "exam",
    "examinations",
    "kcse",
    "kcpe",
    "junior-school",
    "basic-education",
    "scholarship",
    "bursary",
    "admission",
    "admissions",
    "policy",
    "policies",
    "research",
    "training",
    "teacher",
    "teachers",
    "learning",
    "student",
    "students",
    "education-sector"

]

candidate_links = []

for link in education_links:

    lower = link.lower()

    if any(keyword in lower for keyword in ARTICLE_KEYWORDS):

        candidate_links.append(link)

print(f"Candidate education pages: {len(candidate_links)}")

Candidate education pages: 916


*Removing duplicates*

In [12]:
# Remove duplicate links
candidate_links = list(set(candidate_links))

# Keep only valid HTTP/HTTPS links
candidate_links = [

    url for url in candidate_links

    if url.startswith("http")

]

# Sort the links alphabetically
candidate_links.sort()

print(f"Unique education links: {len(candidate_links)}")

Unique education links: 338


*Download Articles*

In [ ]:
from tqdm import tqdm

education_articles = []

for url in tqdm(candidate_links, desc="Downloading Education Articles"):

    soup = fetch_education_page(url)

    if soup is None:
        continue

    # Get the page title
    title = ""

    if soup.title:
        title = soup.title.get_text(strip=True)

    # Extract all paragraph text
    paragraphs = soup.find_all("p")

    text = " ".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
    )

    # Skip pages with very little content
    if len(text) < 200:
        continue

    education_articles.append({
        "url": url,
        "title": title,
        "text": text
    })

print(f"Education articles collected: {len(education_articles)}")


⏱️ Timed out while fetching: http://cbc-conference.go.ke/



❌ Failed to fetch: http://info@universitiesfund.go.ke
Reason: HTTPConnectionPool(host='universitiesfund.go.ke', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x000001C2C68EF6D0>: Failed to resolve 'universitiesfund.go.ke' ([Errno 11001] getaddrinfo failed)"))



❌ Failed to fetch: http://unifund.eujimsolutions.co.ke/policies-guidelines/
Reason: HTTPConnectionPool(host='unifund.eujimsolutions.co.ke', port=80): Max retries exceeded with url: /policies-guidelines/ (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x000001C2C5CFB1D0>: Failed to resolve 'unifund.eujimsolutions.co.ke' ([Errno 11001] getaddrinfo failed)"))



❌ Failed to fetch: http://www.tvetauthority.go.ke/
Reason: HTTPConnectionPool(host='www.tvetauthority.go.ke', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x000001C2C7B1AC50>: Failed to resolve 'www.tvetauthority.go.ke' ([Errno 11001] getaddrinfo failed)"))



⏱️ Timed out while fetching: https://studentportal.ufb.go.ke:8443/


...


❌ Failed to fetch: https://www.education.go.ke/education.go.ke/faqs
Reason: 404 Client Error: Not Found for url: https://www.education.go.ke/education.go.ke/faqs



❌ Failed to fetch: https://www.education.go.ke/www.education.go.ke
Reason: HTTPSConnectionPool(host='www.education.go.ke', port=443): Read timed out.



❌ Failed to fetch: https://www.facebook.com/ExamsCouncil
Reason: 400 Client Error: Bad Request for url: https://www.facebook.com/ExamsCouncil



❌ Failed to fetch: https://www.facebook.com/people/Kenya-Education-Management-Institute-KEMI/100063918345840/
Reason: 400 Client Error: Bad Request for url: https://www.facebook.com/people/Kenya-Education-Management-Institute-KEMI/100063918345840/



❌ Failed to fetch: https://www.facebook.com/universities.fund
Reason: 400 Client Error: Bad Request for url: https://www.facebook.com/universities.fund


In [ ]:
education_articles = []

for url in tqdm(candidate_links, desc="Scraping Education Pages"):
    try:
        soup = fetch_education_page(url)

        if soup is None:
            continue

        # ---------------------------
        # Get Title
        # ---------------------------

        if soup.title:
            title = soup.title.get_text(strip=True)
        else:
            title = ""

        # ---------------------------
        # Get Meta Description
        # ---------------------------

        description = ""

        meta = soup.find("meta", attrs={"name": "description"})

        if meta:
            description = meta.get("content", "")

        # ---------------------------
        # Extract Paragraphs
        # ---------------------------

        paragraphs = soup.find_all("p")

        text = " ".join(
            p.get_text(" ", strip=True)
            for p in paragraphs
        )

        # Skip pages with very little content
        if len(text) < 150:
            continue

        education_articles.append({
            "url": url,
            "title": title,
            "description": description,
            "text": text
        })

    except KeyboardInterrupt:
        print("\n⚠️ Scraping interrupted by user.")
        break

print()
print("Education articles collected:", len(education_articles))

Scraping Education Pages:   0%|          | 1/241 [00:21<1:24:12, 21.05s/it]


❌ Failed to fetch: http://cbc-conference.go.ke/
Reason: HTTPConnectionPool(host='cbc-conference.go.ke', port=80): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x000002C05E620990>, 'Connection to cbc-conference.go.ke timed out. (connect timeout=30)'))

❌ Failed to fetch: http://info@universitiesfund.go.ke
Reason: HTTPConnectionPool(host='universitiesfund.go.ke', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x000002C05EB07150>: Failed to resolve 'universitiesfund.go.ke' ([Errno 11001] getaddrinfo failed)"))

❌ Failed to fetch: http://unifund.eujimsolutions.co.ke/policies-guidelines/
Reason: HTTPConnectionPool(host='unifund.eujimsolutions.co.ke', port=80): Max retries exceeded with url: /policies-guidelines/ (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x000002C05ECEBD50>: Failed to resolve 'unifund.eujimsolutions.co.ke

Scraping Education Pages:   2%|▏         | 5/241 [00:24<15:37,  3.97s/it]  


❌ Failed to fetch: http://www.tvetauthority.go.ke/
Reason: HTTPConnectionPool(host='www.tvetauthority.go.ke', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x000002C05E622DD0>: Failed to resolve 'www.tvetauthority.go.ke' ([Errno 11001] getaddrinfo failed)"))


Scraping Education Pages:  37%|███▋      | 88/241 [02:45<19:50,  7.78s/it]

...

Scraping Education Pages:  37%|███▋      | 89/241 [03:07<29:48, 11.76s/it]


❌ Failed to fetch: https://ufb.go.ke/tenders/
Reason: HTTPSConnectionPool(host='ufb.go.ke', port=443): Max retries exceeded with url: /tenders/ (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000002C05EF387D0>, 'Connection to ufb.go.ke timed out. (connect timeout=30)'))


Scraping Education Pages:  38%|███▊      | 92/241 [04:34<1:13:35, 29.63s/it]


❌ Failed to fetch: https://www.education.go.ke/
Reason: HTTPSConnectionPool(host='www.education.go.ke', port=443): Read timed out.


Scraping Education Pages:  40%|███▉      | 96/241 [06:14<55:35, 23.00s/it]  

Create DataFrames

In [ ]:
import pandas as pd

records = []
for article in education_articles:
    url = article.get("url", "")
    title = article.get("title", "")
    text = article.get("text", "")

    if not url:
        continue

    records.append({
        "ID": hashlib.md5(url.encode("utf-8")).hexdigest(),
        "Website": url.split("//")[-1].split("/")[0] if "//" in url else url,
        "Title": title,
        "Date": article.get("date", ""),
        "PSA": text,
        "URL": url,
    })

if records:
    df = pd.DataFrame(records)
else:
    df = pd.DataFrame(columns=["ID", "Website", "Title", "Date", "PSA", "URL"])

print(df.head())
print(f"Rows in dataframe: {len(df)}")

NameError: name 'education_articles' is not defined